In [0]:
# Databricks notebook source
import unittest
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count

spark = SparkSession.builder.getOrCreate()

# Path to the dataset for integration testing [cite: 20]
sample_path = "/Volumes/ecommerce_analytics_dev/bronze_layer/landing_csvs/2019-Oct.csv"

print(f"✅ Transformation Test Environment Ready for path: {sample_path}")

In [0]:
# 1. Create a broader set of dummy data to show insights
dummy_data = [
    (101, 650.0), # High value
    (102, 150.0), # Low value
    (103, 500.0), # Boundary (should be False)
    (104, 500.01) # Boundary (should be True)
]
df_dummy = spark.createDataFrame(dummy_data, ["user_id", "price"])

# 2. Apply Transformation
transformed_dummy = df_dummy.withColumn("is_high_value", when(col("price") > 500, True).otherwise(False))

# --- BUSINESS INSIGHTS FROM TEST ---
print("📊 DUMMY DATA LOGIC PREVIEW:")
transformed_dummy.show()

class TestTransformationLogic(unittest.TestCase):
    def test_logic_threshold(self):
        """Domain 4.2: Verify the >500 business rule [cite: 143]"""
        res = {row['user_id']: row['is_high_value'] for row in transformed_dummy.collect()}
        self.assertTrue(res[101], "650 should be high value")
        self.assertFalse(res[102], "150 should NOT be high value")
        self.assertFalse(res[103], "Exactly 500 should NOT be high value")

suite_dummy = unittest.TestLoader().loadTestsFromTestCase(TestTransformationLogic)
result_dummy = unittest.TextTestRunner(verbosity=1).run(suite_dummy)

In [0]:
# 1. Read a small slice of the 67M+ record dataset
df_original = spark.read.option("header", "true").csv(sample_path).limit(500)

# --- GENUINE INSIGHTS FROM DATA ---
print("📊 ORIGINAL DATA SAMPLE INSIGHTS:")
print(f"Rows Sampled: {df_original.count()}")
print(f"Columns Found: {df_original.columns}")

class TestOriginalSchema(unittest.TestCase):
    def test_source_columns(self):
        """Domain 3.3: Verify schema inference for critical columns """
        required_cols = ["event_time", "event_type", "product_id", "price", "user_id", "user_session"]
        for c in required_cols:
            self.assertIn(c, df_original.columns, f"CRITICAL: {c} is missing from the source CSV!")

suite_orig = unittest.TestLoader().loadTestsFromTestCase(TestOriginalSchema)
result_orig = unittest.TextTestRunner(verbosity=1).run(suite_orig)

In [0]:
# Consolidated Final Status
all_passed = result_dummy.wasSuccessful() and result_orig.wasSuccessful()
final_status = "✅ LOGIC VALIDATED" if all_passed else "❌ LOGIC FAILURE"

print("-" * 50)
print(f"FINAL TRANSFORMATION SUMMARY: {final_status}")
print("-" * 50)
print(f"Dummy Logic Checks:  {'PASS' if result_dummy.wasSuccessful() else 'FAIL'}")
print(f"Source Schema Check: {'PASS' if result_orig.wasSuccessful() else 'FAIL'}")
print("-" * 50)

if not all_passed:
    raise Exception(f"Pipeline Automation Halted: {final_status}")